# 19 - Multi-Agent Reinforcement Learning

## Learning Objectives
1. Understand Nash equilibrium and independent Q-learning dynamics in matrix games
2. Implement cooperative pursuit (2 agents, 1 target) with CTDE
3. Analyze Nash equilibrium convergence of IQL on Prisoner's Dilemma and RPS
4. Emergent specialization: 3 agents, cooperative task, measure role differentiation


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Packages loaded: numpy', np.__version__)
print('MARL demo: matrix games, cooperative pursuit (CTDE), Nash convergence, specialization')


## Level 1: 2-Player Matrix Games with Independent Q-Learning

Prisoner's Dilemma payoff matrix: R=[[3,0],[5,1]] (row=cooperate=0, defect=1; col=same).
Rock-Paper-Scissors: Nash = uniform mix (1/3 each).
Each agent maintains independent Q-values and updates via REINFORCE-style Q-learning.


In [ ]:
def run_prisoners_dilemma(n_steps: int = 5000, alpha: float = 0.1,
                           eps: float = 0.1) -> tuple:
    """Independent Q-learning on Prisoner's Dilemma.
    Payoff (row player): R[a1][a2]. Nash = (Defect, Defect).
    Actions: 0=Cooperate, 1=Defect.
    """
    # Payoff matrix: R[a_row][a_col] = (reward_agent1, reward_agent2)
    # Based on task spec: R=[[3,0],[5,1]]
    payoff = [
        [(3, 3), (0, 5)],   # row=Cooperate: vs C->(3,3), vs D->(0,5)
        [(5, 0), (1, 1)],   # row=Defect: vs C->(5,0), vs D->(1,1)
    ]
    n_actions = 2

    # Q-tables: stateless single-stage game
    Q1 = np.zeros(n_actions)
    Q2 = np.zeros(n_actions)

    action_hist_1 = []
    reward_hist_1 = []

    for step in range(n_steps):
        # Epsilon-greedy action selection
        a1 = (np.random.randint(n_actions) if np.random.random() < eps
               else np.argmax(Q1))
        a2 = (np.random.randint(n_actions) if np.random.random() < eps
               else np.argmax(Q2))

        r1, r2 = payoff[a1][a2]

        # Independent Q-update (single-step game: no next state)
        Q1[a1] += alpha * (r1 - Q1[a1])
        Q2[a2] += alpha * (r2 - Q2[a2])

        action_hist_1.append(a1)
        reward_hist_1.append(r1)

    return action_hist_1, reward_hist_1, Q1, Q2


def run_rps(n_steps: int = 5000, alpha: float = 0.05,
            eps: float = 0.15) -> tuple:
    """Independent Q-learning on Rock-Paper-Scissors.
    Nash equilibrium = uniform (1/3, 1/3, 1/3).
    Actions: 0=Rock, 1=Paper, 2=Scissors.
    """
    def payoff(a1, a2):
        if a1 == a2:
            return 0, 0
        # 0=Rock beats 2=Scissors, 1=Paper beats 0=Rock, 2=Scissors beats 1=Paper
        wins = {0: 2, 1: 0, 2: 1}
        if wins[a1] == a2:
            return 1, -1
        return -1, 1

    n_actions = 3
    Q1 = np.zeros(n_actions)
    Q2 = np.zeros(n_actions)
    action_hist_1 = []
    reward_hist_1 = []

    for step in range(n_steps):
        a1 = (np.random.randint(n_actions) if np.random.random() < eps
               else np.argmax(Q1))
        a2 = (np.random.randint(n_actions) if np.random.random() < eps
               else np.argmax(Q2))
        r1, r2 = payoff(a1, a2)
        Q1[a1] += alpha * (r1 - Q1[a1])
        Q2[a2] += alpha * (r2 - Q2[a2])
        action_hist_1.append(a1)
        reward_hist_1.append(r1)

    return action_hist_1, reward_hist_1, Q1, Q2


# Run both games
np.random.seed(42)
ah1_pd, rh1_pd, Q1_pd, Q2_pd = run_prisoners_dilemma(n_steps=5000)
np.random.seed(42)
ah1_rps, rh1_rps, Q1_rps, Q2_rps = run_rps(n_steps=5000)

print('Prisoner Dilemma Results:')
print(f'  Agent 1 Q: Cooperate={Q1_pd[0]:.3f}, Defect={Q1_pd[1]:.3f}')
print(f'  Agent 2 Q: Cooperate={Q2_pd[0]:.3f}, Defect={Q2_pd[1]:.3f}')
print(f'  Converged to Nash (Defect,Defect)? '
      f'{np.argmax(Q1_pd)==1 and np.argmax(Q2_pd)==1}')
last_coop = np.mean(np.array(ah1_pd[-1000:]) == 0)
print(f'  Last-1000 cooperation rate (Agent 1): {last_coop:.2f}')

print('Rock-Paper-Scissors Results:')
print(f'  Agent 1 Q: Rock={Q1_rps[0]:.3f}, Paper={Q1_rps[1]:.3f}, Scissors={Q1_rps[2]:.3f}')
action_names = ['Rock', 'Paper', 'Scissors']
print(f'  Agent 1 dominant: {action_names[np.argmax(Q1_rps)]} (Nash=uniform)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
window = 100
coop_ma = np.convolve(np.array(ah1_pd) == 0, np.ones(window) / window, mode='valid')
axes[0].plot(coop_ma, color='#2c7bb6', linewidth=1.5)
axes[0].axhline(y=0.0, color='#d7191c', linestyle='--',
                label='Nash: 0% cooperation (both Defect)')
axes[0].set_title("Prisoner's Dilemma: Cooperation Rate\n(Nash=Defect)", fontsize=11)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Cooperation Rate (rolling avg)')
axes[0].legend(); axes[0].set_facecolor('#f8f8f8'); axes[0].grid(True, alpha=0.4)

ah_arr_rps = np.array(ah1_rps)
for act, name, color in zip([0, 1, 2], action_names, ['#2c7bb6', '#d7191c', '#1a9641']):
    freq_ma = np.convolve(ah_arr_rps == act, np.ones(window) / window, mode='valid')
    axes[1].plot(freq_ma, color=color, label=name, linewidth=1.5)
axes[1].axhline(y=1/3, color='black', linestyle='--', alpha=0.6, label='Nash: 1/3 each')
axes[1].set_title('Rock-Paper-Scissors: Action Frequencies\n(Nash=uniform)', fontsize=11)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Action Frequency (rolling avg)')
axes[1].legend(fontsize=9); axes[1].set_facecolor('#f8f8f8'); axes[1].grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('/tmp/marl_matrix_games.png', dpi=100, bbox_inches='tight')
plt.show()


## Level 2: Cooperative Pursuit with CTDE (2 agents, 1 target, 6x6 grid)

Two agents cooperate to catch a randomly moving target.
CTDE: centralized training (shared reward signal, joint Q-table for critic),
decentralized execution (each agent acts on local obs only).
Compare: independent Q-learning vs CTDE on cooperative pursuit.


In [ ]:
class PursuitEnv:
    """6x6 GridWorld: 2 agents cooperate to catch a moving target."""

    def __init__(self, size: int = 6):
        self.size = size
        self.n_actions = 4  # up, down, left, right
        self.deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def reset(self):
        positions = np.random.choice(self.size ** 2, 3, replace=False)
        self.a1 = (positions[0] // self.size, positions[0] % self.size)
        self.a2 = (positions[1] // self.size, positions[1] % self.size)
        self.target = (positions[2] // self.size, positions[2] % self.size)
        return self._obs()

    def _obs(self):
        return (self.a1[0], self.a1[1], self.a2[0], self.a2[1],
                self.target[0], self.target[1])

    def _move(self, pos, action):
        r, c = pos
        dr, dc = self.deltas[action]
        return (max(0, min(self.size - 1, r + dr)),
                max(0, min(self.size - 1, c + dc)))

    def step(self, a1, a2):
        self.a1 = self._move(self.a1, a1)
        self.a2 = self._move(self.a2, a2)
        # Target moves randomly
        self.target = self._move(self.target, np.random.randint(self.n_actions))
        d1 = abs(self.a1[0] - self.target[0]) + abs(self.a1[1] - self.target[1])
        d2 = abs(self.a2[0] - self.target[0]) + abs(self.a2[1] - self.target[1])
        # Require BOTH agents adjacent: harder task, random policy rarely succeeds
        caught = (d1 <= 1) and (d2 <= 1)
        one_close = (d1 <= 1) or (d2 <= 1)
        reward = 1.0 if caught else (0.1 if one_close else -0.01)
        return self._obs(), reward, caught


def agent_local_obs_idx(obs, agent_id, size=6):
    """Local obs: (agent_r, agent_c, target_r, target_c) -> int index."""
    if agent_id == 0:
        return obs[0] * size**3 + obs[1] * size**2 + obs[4] * size + obs[5]
    return obs[2] * size**3 + obs[3] * size**2 + obs[4] * size + obs[5]


def run_independent_ql(
    n_episodes: int = 800, alpha: float = 0.1, gamma: float = 0.9,
    eps: float = 0.2, size: int = 6
) -> tuple:
    """Independent Q-learning: each agent has its own Q-table."""
    env = PursuitEnv(size=size)
    n_local = size ** 4  # (ar, ac, tr, tc)
    Q1 = np.zeros((n_local, env.n_actions))
    Q2 = np.zeros((n_local, env.n_actions))
    catch_rates = []

    for ep in range(n_episodes):
        obs = env.reset()
        done = False
        t = 0
        while not done and t < 40:
            s1 = agent_local_obs_idx(obs, 0, size)
            s2 = agent_local_obs_idx(obs, 1, size)
            a1 = (np.random.randint(env.n_actions) if np.random.random() < eps
                   else np.argmax(Q1[s1]))
            a2 = (np.random.randint(env.n_actions) if np.random.random() < eps
                   else np.argmax(Q2[s2]))
            obs2, r, done = env.step(a1, a2)
            s1n = agent_local_obs_idx(obs2, 0, size)
            s2n = agent_local_obs_idx(obs2, 1, size)
            # Shared reward, independent updates
            Q1[s1, a1] += alpha * (r + gamma * np.max(Q1[s1n]) - Q1[s1, a1])
            Q2[s2, a2] += alpha * (r + gamma * np.max(Q2[s2n]) - Q2[s2, a2])
            obs = obs2
            t += 1
        catch_rates.append(1.0 if done else 0.0)
    return Q1, Q2, catch_rates


def run_ctde_pursuit(
    n_episodes: int = 800, alpha: float = 0.1, gamma: float = 0.9,
    eps: float = 0.2, size: int = 4
) -> tuple:
    """CTDE: centralized joint Q(s_joint, a1, a2) during training.
    Decentralized execution: each agent acts on local obs.
    Uses dict for Q to handle sparse state space.
    """
    env = PursuitEnv(size=size)
    n_actions = env.n_actions
    n_joint_actions = n_actions ** 2

    def joint_state(obs):
        a1r, a1c, a2r, a2c, tr, tc = obs
        return (a1r * size**5 + a1c * size**4 + a2r * size**3
                + a2c * size**2 + tr * size + tc)

    Q_joint = {}  # dict Q: joint_state -> np.array(n_joint_actions)
    catch_rates = []

    for ep in range(n_episodes):
        obs = env.reset()
        done = False
        t = 0
        while not done and t < 40:
            js = joint_state(obs)
            if js not in Q_joint:
                Q_joint[js] = np.zeros(n_joint_actions)
            ja = (np.random.randint(n_joint_actions) if np.random.random() < eps
                  else np.argmax(Q_joint[js]))
            a1, a2 = ja // n_actions, ja % n_actions
            obs2, r, done = env.step(a1, a2)
            js2 = joint_state(obs2)
            if js2 not in Q_joint:
                Q_joint[js2] = np.zeros(n_joint_actions)
            Q_joint[js][ja] += alpha * (r + gamma * np.max(Q_joint[js2]) - Q_joint[js][ja])
            obs = obs2
            t += 1
        catch_rates.append(1.0 if done else 0.0)
    return Q_joint, catch_rates


print('Training independent Q-learners on pursuit (6x6)...')
np.random.seed(42)
Q1_ind, Q2_ind, rates_ind = run_independent_ql(n_episodes=800, size=6)
print(f'  IQL final 100-ep catch rate: {np.mean(rates_ind[-100:]):.2f}')

print('Training CTDE joint Q on pursuit (4x4)...')
np.random.seed(42)
Q_joint_ctde, rates_ctde = run_ctde_pursuit(n_episodes=800, size=4)
print(f'  CTDE final 100-ep catch rate: {np.mean(rates_ctde[-100:]):.2f}')

# Also IQL on same 4x4 for fair comparison
np.random.seed(42)
_, _, rates_ind_4 = run_independent_ql(n_episodes=800, size=4)
print(f'  IQL (4x4) final 100-ep catch rate: {np.mean(rates_ind_4[-100:]):.2f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
window = 50
cr_ctde = np.convolve(rates_ctde, np.ones(window) / window, mode='valid')
cr_ind4 = np.convolve(rates_ind_4, np.ones(window) / window, mode='valid')

ep_range = np.arange(len(cr_ctde))
axes[0].plot(ep_range, cr_ctde, color='#d7191c', label='CTDE (joint Q, 4x4)', linewidth=2)
axes[0].plot(ep_range, cr_ind4[:len(cr_ctde)], color='#2c7bb6',
             label='Indep. QL (4x4)', linewidth=2)
axes[0].set_xlabel('Episode', fontsize=12)
axes[0].set_ylabel('Catch Rate', fontsize=12)
axes[0].set_title('Cooperative Pursuit: CTDE vs Independent QL', fontsize=12)
axes[0].legend(); axes[0].set_facecolor('#f8f8f8'); axes[0].grid(True, alpha=0.4)

methods_bar = ['IQL (6x6)', 'IQL (4x4)', 'CTDE (4x4)']
perfs_bar = [np.mean(rates_ind[-100:]), np.mean(rates_ind_4[-100:]),
             np.mean(rates_ctde[-100:])]
colors_bar = ['#636363', '#2c7bb6', '#d7191c']
bars = axes[1].bar(methods_bar, perfs_bar, color=colors_bar, alpha=0.85, edgecolor='black')
axes[1].set_ylabel('Catch Rate (last 100 eps)', fontsize=12)
axes[1].set_title('Cooperative Pursuit: Final Performance', fontsize=12)
axes[1].set_ylim(0, 1.1)
axes[1].set_facecolor('#f8f8f8'); axes[1].grid(True, alpha=0.4, axis='y')
for bar, val in zip(bars, perfs_bar):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/marl_pursuit.png', dpi=100, bbox_inches='tight')
plt.show()


## Real-World Example 1: Nash Equilibrium Convergence Analysis

Check if IQL converges to Nash on Prisoner's Dilemma and Rock-Paper-Scissors.
Nash for PD: (Defect, Defect). Nash for RPS: uniform (1/3, 1/3, 1/3).
Plot action distribution evolution and compute Nash deviation (regret).


In [ ]:
def measure_nash_convergence(
    action_hist: list, nash_action: int = None,
    nash_dist: np.ndarray = None, window: int = 200
) -> np.ndarray:
    """Measure distance to Nash equilibrium over training.
    For PD: convergence = P(defect) -> 1.
    For RPS: convergence = |empirical_freq - 1/3| -> 0.
    """
    n_steps = len(action_hist)
    hist_arr = np.array(action_hist)
    n_actions = len(np.unique(hist_arr))
    deviations = []

    for t in range(window, n_steps, window // 4):
        window_acts = hist_arr[max(0, t - window):t]
        empirical = np.array([(window_acts == a).mean() for a in range(n_actions)])
        if nash_dist is not None:
            # Nash distance: L1 distance from Nash distribution
            dev = np.sum(np.abs(empirical - nash_dist))
        elif nash_action is not None:
            # PD: how far is P(nash_action) from 1.0
            dev = 1.0 - empirical[nash_action]
        else:
            dev = 0.0
        deviations.append(dev)

    return np.array(deviations)


# Sweep alpha values to show convergence speed
alphas = [0.01, 0.05, 0.1, 0.3]
pd_convergence = {}
rps_convergence = {}

for alpha_val in alphas:
    np.random.seed(42)
    ah_pd, _, _, _ = run_prisoners_dilemma(n_steps=5000, alpha=alpha_val, eps=0.05)
    # Nash PD: defect (action=1)
    pd_convergence[alpha_val] = measure_nash_convergence(
        ah_pd, nash_action=1, window=200
    )

    np.random.seed(42)
    ah_rps, _, _, _ = run_rps(n_steps=5000, alpha=alpha_val, eps=0.05)
    # Nash RPS: uniform (1/3, 1/3, 1/3)
    rps_convergence[alpha_val] = measure_nash_convergence(
        ah_rps, nash_dist=np.array([1/3, 1/3, 1/3]), window=200
    )

print('Nash Convergence Summary:')
print(f'{"alpha":<8} {"PD final dev":<15} {"RPS final dev":<15}')
print('-' * 38)
for alpha_val in alphas:
    pd_dev = pd_convergence[alpha_val][-1] if len(pd_convergence[alpha_val]) > 0 else float('nan')
    rps_dev = rps_convergence[alpha_val][-1] if len(rps_convergence[alpha_val]) > 0 else float('nan')
    print(f'{alpha_val:<8.2f} {pd_dev:<15.4f} {rps_dev:<15.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors_alpha = plt.cm.viridis(np.linspace(0.1, 0.9, len(alphas)))

for alpha_val, color in zip(alphas, colors_alpha):
    pd_d = pd_convergence[alpha_val]
    rps_d = rps_convergence[alpha_val]
    x_pd = np.linspace(0, 5000, len(pd_d))
    x_rps = np.linspace(0, 5000, len(rps_d))
    axes[0].plot(x_pd, pd_d, color=color, label=f'alpha={alpha_val}', linewidth=1.5)
    axes[1].plot(x_rps, rps_d, color=color, label=f'alpha={alpha_val}', linewidth=1.5)

axes[0].set_title("Prisoner's Dilemma: Nash Deviation over Time", fontsize=11)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('P(Defect) deviation from Nash=1.0')
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.5, label='Nash equilibrium')
axes[0].legend(fontsize=9); axes[0].set_facecolor('#f8f8f8'); axes[0].grid(True, alpha=0.4)

axes[1].set_title('RPS: Nash Deviation (L1 from uniform)', fontsize=11)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('L1 distance from (1/3, 1/3, 1/3)')
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5, label='Nash equilibrium')
axes[1].legend(fontsize=9); axes[1].set_facecolor('#f8f8f8'); axes[1].grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('/tmp/marl_nash_convergence.png', dpi=100, bbox_inches='tight')
plt.show()


## Real-World Example 2: Communication Protocol

Two agents must coordinate to reach different targets.
Agent 1 knows target A location; Agent 2 knows target B location.
They share a binary message; show whether an emergent protocol develops.


In [ ]:
def run_communication_game(
    n_episodes: int = 1000, alpha: float = 0.1, eps: float = 0.2
) -> tuple:
    """Communication game: agents learn to coordinate via emergent protocol.
    Agent1 observes target_A side (0=left, 1=right), sends binary message m.
    Agent2 receives m, observes target_B, acts.
    Joint reward = 1 only when both agents go to the correct side.
    """
    n_messages = 2
    n_actions = 2  # 0=left, 1=right

    # Q1[target_A, (msg, action)]: Agent1 combines message and action
    Q1 = np.zeros((2, n_messages * n_actions))
    # Q2[msg, target_B, action]
    Q2 = np.zeros((n_messages, 2, n_actions))

    success_rates = []
    msg_informativeness = []  # does message encode target_A correctly?

    for ep in range(n_episodes):
        target_A = np.random.randint(2)
        target_B = np.random.randint(2)

        # Agent1 picks combined (msg, action)
        combined = (np.random.randint(n_messages * n_actions)
                    if np.random.random() < eps else np.argmax(Q1[target_A]))
        m = combined // n_actions
        a1 = combined % n_actions

        # Agent2 receives m, picks action
        a2 = (np.random.randint(n_actions) if np.random.random() < eps
               else np.argmax(Q2[m, target_B]))

        # Joint reward
        r = 1.0 if (a1 == target_A and a2 == target_B) else 0.0

        Q1[target_A, combined] += alpha * (r - Q1[target_A, combined])
        Q2[m, target_B, a2] += alpha * (r - Q2[m, target_B, a2])

        success_rates.append(r)
        # Message is informative if it consistently encodes target_A
        msg_informativeness.append(1.0 if m == target_A else 0.0)

    return success_rates, msg_informativeness, Q1, Q2


np.random.seed(42)
comm_success, msg_info, Q1_comm, Q2_comm = run_communication_game(n_episodes=1500)

window = 50
succ_ma = np.convolve(comm_success, np.ones(window) / window, mode='valid')
msg_ma = np.convolve(msg_info, np.ones(window) / window, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x = np.arange(len(succ_ma))

axes[0].plot(x, succ_ma, color='#2c7bb6', linewidth=2)
axes[0].axhline(y=0.25, color='gray', linestyle='--', label='Random baseline (0.25)')
axes[0].axhline(y=1.0, color='#1a9641', linestyle=':', label='Optimal (1.0)')
axes[0].set_xlabel('Episode', fontsize=12)
axes[0].set_ylabel('Joint Success Rate', fontsize=12)
axes[0].set_title('Communication Game: Coordination Learning', fontsize=12)
axes[0].legend(); axes[0].set_facecolor('#f8f8f8'); axes[0].grid(True, alpha=0.4)

axes[1].plot(x, msg_ma, color='#d7191c', linewidth=2, label='Message=Target')
axes[1].axhline(y=0.5, color='gray', linestyle='--', label='Random (0.5)')
axes[1].axhline(y=1.0, color='#1a9641', linestyle=':', label='Perfect protocol (1.0)')
axes[1].set_xlabel('Episode', fontsize=12)
axes[1].set_ylabel('P(message encodes target_A)', fontsize=12)
axes[1].set_title('Emergent Communication Protocol', fontsize=12)
axes[1].legend(); axes[1].set_facecolor('#f8f8f8'); axes[1].grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('/tmp/marl_communication.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Final joint success rate: {np.mean(comm_success[-100:]):.2f}')
print(f'Message encodes target correctly (last 100): {np.mean(msg_info[-100:]):.2f}')


## Real-World Example 3: Emergent Specialization (3 Agents)

3 cooperative agents on 9x9 grid, 3 fixed targets at corners.
Role-differentiated reward: each agent gets 3x reward for its 'home' target.
This creates role specialization: agents optimize for different subtasks.
Specialization entropy: 0 = fully specialized, ln(3)=1.10 = no specialization.


In [ ]:
def run_specialization_3agents(
    n_agents: int = 3, size: int = 9,
    n_episodes: int = 800, alpha: float = 0.2,
    gamma: float = 0.9, eps: float = 0.15
) -> tuple:
    """3 agents, 3 fixed targets with role-differentiated rewards.
    Agent i gets 3x reward for its home target (targets[i]) vs 1x for others.
    Tracks: collection efficiency and role specialization (entropy per agent).
    """
    n_actions = 4
    deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    n_targets = n_agents

    # Targets spread across the grid: corners and center-bottom
    targets = [(0, 0), (0, size - 1), (size - 1, size // 2)]
    home_reward = 3.0  # each agent gets 3x reward for its home target

    def move(pos, action):
        r, c = pos
        dr, dc = deltas[action]
        return (max(0, min(size - 1, r + dr)), max(0, min(size - 1, c + dc)))

    # Q-table per agent: (pos, target_idx) -> action
    n_pos = size ** 2
    Q_all = [np.zeros((n_pos, n_targets, n_actions)) for _ in range(n_agents)]

    collection_history = []  # targets collected per episode
    specialization_history = [[] for _ in range(n_agents)]  # which target each collected

    for ep in range(n_episodes):
        # Start agents near their home targets to encourage specialization
        used_pos = set()
        agent_pos = []
        for i in range(n_agents):
            tr, tc = targets[i]
            for _ in range(100):  # try near home target first
                r = max(0, min(size - 1, tr + np.random.randint(-3, 4)))
                c = max(0, min(size - 1, tc + np.random.randint(-3, 4)))
                p = (r, c)
                if p not in used_pos and p not in targets:
                    used_pos.add(p)
                    agent_pos.append(p)
                    break
            else:
                # Fallback: random position
                while True:
                    p = (np.random.randint(size), np.random.randint(size))
                    if p not in used_pos and p not in targets:
                        used_pos.add(p)
                        agent_pos.append(p)
                        break

        targets_collected = set()
        episode_collections = [None] * n_agents

        for step in range(60):
            if len(targets_collected) == n_targets:
                break

            for i in range(n_agents):
                if episode_collections[i] is not None:
                    continue

                r_i, c_i = agent_pos[i]
                # Find nearest uncollected target
                best_t, best_dist = -1, 9999
                for t_idx, (tr, tc) in enumerate(targets):
                    if t_idx not in targets_collected:
                        d = abs(r_i - tr) + abs(c_i - tc)
                        if d < best_dist:
                            best_dist = d
                            best_t = t_idx
                if best_t == -1:
                    continue

                s = r_i * size + c_i
                a = (np.random.randint(n_actions) if np.random.random() < eps
                      else np.argmax(Q_all[i][s, best_t]))
                new_pos = move(agent_pos[i], a)
                s2 = new_pos[0] * size + new_pos[1]

                if new_pos == targets[best_t] and best_t not in targets_collected:
                    # Home target reward: 3x to encourage specialization
                    r_val = home_reward if best_t == i else 1.0
                    targets_collected.add(best_t)
                    episode_collections[i] = best_t
                else:
                    r_val = -0.01

                Q_all[i][s, best_t, a] += alpha * (
                    r_val + gamma * np.max(Q_all[i][s2, best_t]) - Q_all[i][s, best_t, a]
                )
                agent_pos[i] = new_pos

        collection_history.append(len(targets_collected))
        for i in range(n_agents):
            if episode_collections[i] is not None:
                specialization_history[i].append(episode_collections[i])

    return collection_history, specialization_history


np.random.seed(42)
print('Running role-specialized 3 agents on 9x9 grid (800 episodes)...')
coll_hist, spec_hist = run_specialization_3agents(n_agents=3, size=9, n_episodes=800)
print(f'Final 100-ep mean targets collected: {np.mean(coll_hist[-100:]):.2f} / 3')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

window = 30
coll_ma = np.convolve(coll_hist, np.ones(window) / window, mode='valid')
axes[0].plot(coll_ma, color='#2c7bb6', linewidth=2)
axes[0].axhline(y=3, color='#1a9641', linestyle='--', label='Max (3 targets)')
axes[0].set_xlabel('Episode', fontsize=12)
axes[0].set_ylabel('Targets Collected per Episode', fontsize=12)
axes[0].set_title('3-Agent Cooperative Collection: Learning Curve', fontsize=12)
axes[0].legend(); axes[0].set_facecolor('#f8f8f8'); axes[0].grid(True, alpha=0.4)

# Specialization heatmap: agent x target
spec_matrix = np.zeros((3, 3))
for i, hist in enumerate(spec_hist):
    if hist:
        last_half = hist[len(hist) // 2:]
        for t in last_half:
            spec_matrix[i, t] += 1
        if spec_matrix[i].sum() > 0:
            spec_matrix[i] /= spec_matrix[i].sum()

im = axes[1].imshow(spec_matrix, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=axes[1])
axes[1].set_xticks([0, 1, 2])
axes[1].set_xticklabels(['T0 (0,0)', 'T1 (0,8)', 'T2 (8,4)'], fontsize=10)
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels([f'Agent {i}' for i in range(3)])
axes[1].set_title('Emergent Specialization (late training)', fontsize=11)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{spec_matrix[i,j]:.2f}', ha='center', va='center',
                     fontsize=11, color='white' if spec_matrix[i, j] > 0.5 else 'black')
plt.tight_layout()
plt.savefig('/tmp/marl_specialization.png', dpi=100, bbox_inches='tight')
plt.show()

# Compute specialization entropy (low entropy = high specialization)
def specialization_entropy(dist):
    d = dist + 1e-9
    return -np.sum(d * np.log(d))

print('Agent specialization entropy (lower=more specialized):')
for i in range(3):
    ent = specialization_entropy(spec_matrix[i])
    dominant_target = np.argmax(spec_matrix[i])
    print(f'  Agent {i}: entropy={ent:.3f}, dominant target={dominant_target}')


## Comparison: IQL vs CTDE vs Random on Cooperative Pursuit

Summary of performance across all multi-agent methods.


In [ ]:
# Run random baseline
def run_random_pursuit(n_episodes: int = 800, size: int = 4) -> list:
    """Random policy baseline on pursuit task."""
    env = PursuitEnv(size=size)
    catch_rates = []
    for ep in range(n_episodes):
        env.reset()
        done = False
        for _ in range(40):
            _, _, done = env.step(
                np.random.randint(env.n_actions),
                np.random.randint(env.n_actions),
            )
            if done:
                break
        catch_rates.append(1.0 if done else 0.0)
    return catch_rates


np.random.seed(42)
rates_random = run_random_pursuit(n_episodes=800, size=4)

methods_cmp = ['Random', 'IQL (4x4)', 'CTDE (4x4)']
perfs_cmp = [
    float(np.mean(rates_random[-100:])),
    float(np.mean(rates_ind_4[-100:])),
    float(np.mean(rates_ctde[-100:])),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors_cmp = ['#636363', '#2c7bb6', '#d7191c']
window = 50
cr_random = np.convolve(rates_random, np.ones(window) / window, mode='valid')

ep_range = np.arange(len(cr_ctde))
axes[0].plot(ep_range, cr_ctde, color='#d7191c', label='CTDE', linewidth=2)
axes[0].plot(ep_range, cr_ind4[:len(cr_ctde)], color='#2c7bb6',
             label='IQL', linewidth=2)
axes[0].plot(ep_range, cr_random[:len(cr_ctde)], color='#636363',
             label='Random', linewidth=2, linestyle='--')
axes[0].set_xlabel('Episode', fontsize=12)
axes[0].set_ylabel('Catch Rate', fontsize=12)
axes[0].set_title('Cooperative Pursuit: IQL vs CTDE vs Random', fontsize=12)
axes[0].legend(); axes[0].set_facecolor('#f8f8f8'); axes[0].grid(True, alpha=0.4)

bars = axes[1].bar(methods_cmp, perfs_cmp, color=colors_cmp, alpha=0.85, edgecolor='black')
axes[1].set_ylabel('Catch Rate (last 100 episodes)', fontsize=12)
axes[1].set_title('Final Performance Comparison', fontsize=12)
axes[1].set_ylim(0, 1.1)
axes[1].set_facecolor('#f8f8f8'); axes[1].grid(True, alpha=0.4, axis='y')
for bar, val in zip(bars, perfs_cmp):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/marl_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('Multi-Agent Performance Summary:')
print(f'{"Method":<15} {"Catch Rate":<12}')
print('-' * 28)
for m, p in zip(methods_cmp, perfs_cmp):
    print(f'{m:<15} {p:<12.3f}')

# Nash equilibrium convergence summary
print('\nNash Convergence (last 100 steps):')
pd_last_coop = 1 - np.mean(np.array(ah1_pd[-100:]) == 1)  # deviation from Defect
rps_last_dist = np.array([np.mean(np.array(ah1_rps[-500:]) == a) for a in range(3)])
rps_nash_dev = np.sum(np.abs(rps_last_dist - 1/3))
print(f'  PD: deviation from Nash (Defect,Defect) = {pd_last_coop:.3f}')
print(f'  RPS: L1 distance from uniform Nash = {rps_nash_dev:.3f}')
print(f'  PD Nash reached: {pd_last_coop < 0.1}')
print(f'  RPS Nash reached (deviation < 0.1): {rps_nash_dev < 0.1}')


## Key Takeaways

**Core idea:** Multi-agent RL requires agents to handle non-stationarity — as each
agent's policy changes, the environment changes for all others. CTDE addresses this
by using centralized critics during training while keeping execution decentralized.

**Variants and when to use:**

| Method | Non-stationarity | Scales to N | Nash guarantee | Use when |
|--------|-----------------|-------------|----------------|----------|
| Independent QL | Ignored | Yes (large N) | No | Simple cooperative |
| CTDE (MADDPG/QMIX) | Handled | ~10 agents | Approximate | Cooperative, known N |
| Self-play | Symmetric | Yes | Yes (2-player) | Competitive/symmetric |
| Communication | Explicit | ~20 agents | No | Partial observability |

**Common failure modes:**
- IQL cycles in competitive games (non-stationarity)
- CTDE joint Q exponential in N: O(A^N) joint actions
- Credit assignment: agents claim shared reward for others' work


## Exercises

1. **PD with communication**: Add 1-bit pre-game message to Prisoner's Dilemma.
   Can agents escape Nash (Defect,Defect) with communication?
2. **CTDE scaling**: Run `run_ctde_pursuit` on 5x5 grid vs 4x4.
   How does dict Q size scale? At what grid size does memory become limiting?
3. **RPS non-convergence**: Increase alpha to 0.5 in RPS. Show that high learning
   rates cause cycling rather than convergence to the Nash mix.
4. **Specialization without eps decay**: Run `run_specialization_3agents` with
   fixed eps=0.0 (greedy). Does specialization still emerge, or do agents get stuck?
